# 03 — Diarization

**Purpose:** Assign "who spoke when" timestamps to every segment in the audio. This defines the per-speaker chunks that ASR and TTS operate on.

## Two paths
1. **Load external** — your pre-computed diarization from the Postudio platform. Fast, no compute. Set `EXTERNAL_DIARIZATION_PATH` and skip the OSS cells.
2. **Run fresh** — evaluate 3 OSS models locally. Use this to benchmark the platform output or when no external file is available.

## Model selection rationale

| Model | Architecture | Why included |
|---|---|---|
| **Pyannote 3.1** | Segmentation + speaker embedding ECAPA-TDNN | 2024 SOTA; modular design; best-maintained; handles overlapping speech |
| **Pyannote 3.1 (N speakers)** | Same, but speaker count constrained | Reduces confusion errors when you know the cast size |
| **NeMo Sortformer** | Sorting-based end-to-end neural diarizer | NVIDIA; strong on multi-speaker, noisy scenarios; 4-speaker cap (flag) |
| **WhisperX** | Whisper ASR + Pyannote diarization | Gets ASR + diarization in one pass; convenient but less accurate diarization than standalone Pyannote |

## Metric rationale
- **DER (Diarization Error Rate)** = (missed speech + false alarm + speaker confusion) / total speech. Industry standard. We don't have ground truth so DER is N/A, but we compute the formula and you can plug in ground truth later.
- **Speaker count match** vs. actual (manual ground truth from watching the video).
- **Segment coverage %** = how much of the audio has an assigned speaker (vs. silence or missed speech).

## Ensemble: DOVER-Lap voting
When running multiple systems, DOVER-Lap (Directed Overlap-aware DER rEduction and Lap voting) can vote across Pyannote + NeMo + WhisperX outputs and produce a consensus that beats any single system.

**API keys needed:** `HF_TOKEN` (for Pyannote model download). See `API_KEYS.md`.

**Input:** `intermediate/stems/vocals.wav`  
**Output:** `intermediate/diarization/diarization.json`  
         `intermediate/diarization/diarization.rttm`


In [ ]:
import importlib, subprocess, sys
_need = [pkg for pkg, mod in [
    ('pyannote.audio', 'pyannote.audio'),
    ('tqdm',           'tqdm'),
    ('pandas',         'pandas'),
    ('librosa',        'librosa'),
    ('soundfile',      'soundfile'),
    ('matplotlib',     'matplotlib'),
] if not importlib.util.find_spec(mod)]
if _need:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + _need)
print('Core deps ready.')
print('NeMo (optional, large): pip install nemo_toolkit[asr]')
print('DOVER-Lap (optional):   pip install dover-lap')

In [ ]:
import sys, os, json, time
os.environ['PATH'] = '/opt/homebrew/bin:' + os.environ.get('PATH', '')
from getpass import getpass
import torch
import librosa
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.notebook import tqdm
from IPython.display import display

sys.path.insert(0, os.path.abspath('..'))
from config import STEMS_DIR, DIARIZATION_DIR, AUDIO_EXTRACTED_DIR, WHISPER_LANGUAGE_CODE, SOURCE_LANGUAGE, get_torch_device

VOCALS_WAV = os.path.join(STEMS_DIR,      'vocals.wav')
OUT_JSON   = os.path.join(DIARIZATION_DIR, 'diarization.json')
OUT_RTTM   = os.path.join(DIARIZATION_DIR, 'diarization.rttm')

# Read duration from meta.json (fast; avoids loading the full WAV)
_meta_path = os.path.join(AUDIO_EXTRACTED_DIR, 'meta.json')
with open(_meta_path) as _f:
    DURATION = json.load(_f)['duration_seconds']
print(f'Vocals: {DURATION:.1f}s')

# Device detection: CUDA > MPS (Apple Silicon) > CPU
DEVICE, DTYPE = get_torch_device()
COMPUTE_TYPE  = 'float16' if DEVICE in ('cuda', 'mps') else 'int8'
print(f'Device: {DEVICE}  dtype: {DTYPE}  whisperx_compute: {COMPUTE_TYPE}')

all_results  = {}    # model_name -> [{speaker, start, end}, ...]
NUM_SPEAKERS = None  # override in cell-pyannote-n; defined here so cell-winner never raises NameError

HF_TOKEN = os.getenv('HF_TOKEN') or getpass('HuggingFace token (for Pyannote/WhisperX): ')
print('HF token set.')

## Path A — Load External Diarization

Paste the path to your RTTM or JSON file from the Postudio platform. If set, the platform diarization is automatically added as the `External` entry and used as default winner.


In [ ]:
EXTERNAL_DIARIZATION_PATH = ''  # <-- paste full path to your .rttm or .json

def parse_rttm(path):
    segs = []
    with open(path) as f:
        for line in f:
            p = line.strip().split()
            if len(p) < 8 or p[0] != 'SPEAKER': continue
            s, d = float(p[3]), float(p[4])
            segs.append({'speaker': p[7], 'start': s, 'end': s + d})
    return sorted(segs, key=lambda x: x['start'])

def parse_json_diarization(path):
    with open(path) as f: data = json.load(f)
    segs = []
    for d in data:
        segs.append({
            'speaker': d.get('speaker', d.get('label', 'SPEAKER_00')),
            'start':   float(d.get('start', d.get('start_time', 0))),
            'end':     float(d.get('end',   d.get('end_time', 0))),
        })
    return sorted(segs, key=lambda x: x['start'])

if EXTERNAL_DIARIZATION_PATH and os.path.exists(EXTERNAL_DIARIZATION_PATH):
    ext = os.path.splitext(EXTERNAL_DIARIZATION_PATH)[1].lower()
    external_segs = parse_rttm(EXTERNAL_DIARIZATION_PATH) if ext == '.rttm' else parse_json_diarization(EXTERNAL_DIARIZATION_PATH)
    all_results['External'] = external_segs
    speakers = sorted(set(s['speaker'] for s in external_segs))
    print(f'External: {len(external_segs)} segments, {len(speakers)} speakers: {speakers}')
else:
    print('No external file. Run OSS models below.')


## Path B — OSS Models


In [ ]:
# ── Pyannote 3.1 — default (auto speaker count) ───────────────────────────────
try:
    from pyannote.audio import Pipeline

    print(f'Loading Pyannote 3.1 on {DEVICE}...')
    t0 = time.time()
    pipeline_pya = Pipeline.from_pretrained(
        'pyannote/speaker-diarization-3.1', use_auth_token=HF_TOKEN
    ).to(torch.device(DEVICE))
    print(f'Loaded in {time.time()-t0:.1f}s')

    t0 = time.time()
    diarization = pipeline_pya(VOCALS_WAV)
    elapsed = time.time() - t0

    segs = sorted(
        [{'speaker': spk, 'start': turn.start, 'end': turn.end}
         for turn, _, spk in diarization.itertracks(yield_label=True)],
        key=lambda x: x['start']
    )
    all_results['Pyannote-3.1'] = segs
    speakers = sorted(set(s['speaker'] for s in segs))
    print(f'Pyannote-3.1: {len(segs)} segs, {len(speakers)} speakers, {elapsed:.1f}s  RTF={elapsed/DURATION:.2f}')
except Exception as e:
    print(f'Pyannote-3.1 FAILED: {e}')

In [ ]:
# ── Pyannote 3.1 — with known speaker count ───────────────────────────────────
# Constraining the speaker count reduces confusion errors when you know the cast.
# Watch/listen to the first 2 minutes and count distinct speakers.
NUM_SPEAKERS = None  # e.g. 3 — set this after previewing the source

if NUM_SPEAKERS and 'pipeline_pya' in dir():
    try:
        t0 = time.time()
        diar_n = pipeline_pya(VOCALS_WAV, num_speakers=NUM_SPEAKERS)
        elapsed = time.time() - t0
        segs = sorted(
            [{'speaker': spk, 'start': turn.start, 'end': turn.end}
             for turn, _, spk in diar_n.itertracks(yield_label=True)],
            key=lambda x: x['start']
        )
        all_results[f'Pyannote-3.1-N{NUM_SPEAKERS}'] = segs
        print(f'Pyannote (N={NUM_SPEAKERS}): {len(segs)} segs, {elapsed:.1f}s')
    except Exception as e:
        print(f'Pyannote (N={NUM_SPEAKERS}) FAILED: {e}')
else:
    print('Set NUM_SPEAKERS (int) and ensure Pyannote model is loaded to run this.')


In [ ]:
# ── WhisperX — ASR + diarization in one pass ──────────────────────────────────
try:
    import whisperx

    lang_code = WHISPER_LANGUAGE_CODE.get(SOURCE_LANGUAGE, SOURCE_LANGUAGE)
    print(f'WhisperX: lang={lang_code}, device={DEVICE}, compute={COMPUTE_TYPE}')
    t0 = time.time()

    model_wx  = whisperx.load_model('large-v3', DEVICE, compute_type=COMPUTE_TYPE, language=lang_code)
    audio_wx  = whisperx.load_audio(VOCALS_WAV)
    result    = model_wx.transcribe(audio_wx, batch_size=16, language=lang_code)

    model_a, metadata = whisperx.load_align_model(language_code=lang_code, device=DEVICE)
    result = whisperx.align(result['segments'], model_a, metadata, audio_wx, DEVICE)

    diarize_model = whisperx.DiarizationPipeline(use_auth_token=HF_TOKEN, device=DEVICE)
    diarize_segs  = diarize_model(audio_wx)
    result        = whisperx.assign_word_speakers(diarize_segs, result)

    elapsed = time.time() - t0
    segs = sorted(
        [{'speaker': s.get('speaker', 'UNKNOWN'), 'start': s['start'], 'end': s['end'], 'text': s.get('text', '')}
         for s in result['segments']],
        key=lambda x: x['start']
    )
    all_results['WhisperX'] = segs
    print(f'WhisperX: {len(segs)} segs, {elapsed:.1f}s  RTF={elapsed/DURATION:.2f}')
except Exception as e:
    print(f'WhisperX FAILED: {e}')

In [ ]:
# ── NeMo Sortformer ───────────────────────────────────────────────────────────
# NVIDIA's end-to-end diarizer. Strong on overlapping, noisy speech.
# ⚠ NOTE: Sortformer 4spk-v1 has a hard 4-speaker cap. If the source has 5+
#   distinct speakers it will silently merge them — always check speaker count.
try:
    from nemo.collections.asr.models import SortformerEncLabelModel
    t0 = time.time()
    model_nemo = SortformerEncLabelModel.from_pretrained('nvidia/diar_sortformer_4spk-v1')
    output = model_nemo.diarize(audio_filepaths=[VOCALS_WAV])
    elapsed = time.time() - t0
    segs = sorted(
        [{'speaker': item['speaker'], 'start': item['start_sec'], 'end': item['end_sec']}
         for item in output[0]],
        key=lambda x: x['start']
    )
    n_speakers = len(set(s['speaker'] for s in segs))
    if n_speakers >= 4:
        print(f'WARNING: {n_speakers} speakers detected — at NeMo 4-speaker cap. Some may be merged.')
    all_results['NeMo-Sortformer'] = segs
    print(f'NeMo Sortformer: {len(segs)} segs, {n_speakers} speakers, {elapsed:.1f}s')
except ImportError:
    print('NeMo not installed. Run: pip install nemo_toolkit[asr]  (large install, ~4 GB)')
except Exception as e:
    print(f'NeMo FAILED: {e}')


## OSS — MOSS-Audio: joint speaker + emotion in one pass

MOSS-Audio is an audio understanding foundation model — it identifies speakers, emotional states, and events simultaneously without switching between separate systems. This consolidates what would be separate diarization + emotion classification steps.

**Why it's interesting here:**
- Speaker + emotion in a single forward pass — potentially replaces both Pyannote and the librosa emotion heuristic in notebook 05
- Time-aware architecture (time-marker pretraining) → timestamps are first-class, not post-processed
- Emotion labels (neutral / happy / sad / angry / fearful / surprised) can feed directly into the translation WPM adjustment

**Honest caveats:**
- MOSS-Audio is an **understanding model**, not a dedicated diarizer — speaker boundary precision is likely lower than Pyannote
- Prompts the LLM to output a JSON timeline; response parsing can fail if the model reformats output
- **Tamil/Hindi performance is unknown** — benchmarks are English and Mandarin only
- Released April 2026, no production track record; treat as experimental
- Requires GPU: ~10 GB VRAM (4B), ~20 GB (8B)

**Outputs:** `all_results['MOSS-Audio']` (standard format) + `moss_emotions` dict (speaker → emotion) available for notebook 05  
**GitHub:** https://github.com/OpenMOSS/MOSS-Audio

In [ ]:
# ── MOSS-Audio — joint speaker diarization + emotion (experimental) ────────────
# Processes audio in 60-second windows and prompts the model for a JSON timeline.
# If JSON parsing fails, check the raw output printed below and adjust the prompt.
import re as _re
import json as _json

MOSS_MODEL_ID   = 'OpenMOSS-Team/MOSS-Audio-4B-Instruct'
MOSS_CHUNK_SECS = 60  # seconds per chunk; longer = more speaker context

MOSS_DIAR_PROMPT = (
    'Listen carefully to all speakers in this audio. '
    'Return a JSON array where each element represents one speaker turn with these keys: '
    '"start" (float, seconds from the beginning of this clip), '
    '"end" (float, seconds), '
    '"speaker" (string, e.g. "SPEAKER_0" — use consistent IDs across turns), '
    '"emotion" (string, one of: neutral, happy, sad, angry, fearful, surprised). '
    'Return ONLY the JSON array. No extra text, no markdown fences.'
)

try:
    import torch
    from transformers import AutoModel, AutoProcessor

    device = ('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Loading {MOSS_MODEL_ID} on {device} (first run downloads ~5-10 GB)...')
    t0 = time.time()

    moss_model = AutoModel.from_pretrained(
        MOSS_MODEL_ID,
        trust_remote_code=True,
        torch_dtype=(torch.float16 if device == 'mps' else torch.bfloat16 if device == 'cuda' else torch.float32),
        device_map='auto',
    )
    moss_model.eval()

    moss_proc = AutoProcessor.from_pretrained(
        MOSS_MODEL_ID,
        trust_remote_code=True,
        enable_time_marker=True,
    )
    mel_sr = getattr(getattr(moss_proc, 'config', None), 'mel_sr', 16000)
    print(f'Loaded in {time.time()-t0:.1f}s  |  mel_sr={mel_sr} Hz')

    y_moss_full, _ = librosa.load(VOCALS_WAV, sr=mel_sr, mono=True)

    raw_segs = []  # collect all parsed segments across chunks
    t0 = time.time()

    for chunk_start in range(0, int(DURATION), MOSS_CHUNK_SECS):
        chunk_end = min(chunk_start + MOSS_CHUNK_SECS, int(DURATION))
        s_idx = int(chunk_start * mel_sr)
        e_idx = int(chunk_end   * mel_sr)
        chunk = y_moss_full[s_idx:e_idx]

        inputs = moss_proc(text=MOSS_DIAR_PROMPT, audios=[chunk], return_tensors='pt')
        inputs = {k: v.to(moss_model.device) for k, v in inputs.items()}
        if inputs.get('audio_data') is not None:
            inputs['audio_data'] = inputs['audio_data'].to(moss_model.dtype)
        inputs['audio_input_mask'] = inputs['input_ids'] == moss_proc.audio_token_id

        with torch.no_grad():
            gen_ids = moss_model.generate(
                **inputs,
                max_new_tokens=1024,
                do_sample=False,
                use_cache=True,
            )

        resp = moss_proc.decode(
            gen_ids[0, inputs['input_ids'].shape[1]:],
            skip_special_tokens=True,
        ).strip()

        # Extract the JSON array from the response (model may add preamble)
        json_match = _re.search(r'\[.*?\]', resp, _re.DOTALL)
        if not json_match:
            print(f'  [chunk {chunk_start}s] No JSON array found. Raw: {resp[:200]}')
            continue

        try:
            parsed = _json.loads(json_match.group())
        except Exception as parse_err:
            print(f'  [chunk {chunk_start}s] JSON parse error: {parse_err}. Raw: {resp[:200]}')
            continue

        for item in parsed:
            raw_segs.append({
                'speaker': 'MOSS_' + str(item.get('speaker', 'SPEAKER_0')).upper(),
                'start':   float(item.get('start', 0.0)) + chunk_start,
                'end':     float(item.get('end',   0.0)) + chunk_start,
                'emotion': item.get('emotion', 'neutral'),
            })

        print(f'  Chunk {chunk_start}-{chunk_end}s: {len(parsed)} turns parsed')

    elapsed = time.time() - t0

    # Sort and remove duplicates from overlapping chunk windows
    raw_segs.sort(key=lambda x: x['start'])
    segs_moss_diar = []
    for seg in raw_segs:
        if segs_moss_diar and seg['start'] < segs_moss_diar[-1]['end'] - 1.0:
            continue  # likely a duplicate from the chunk overlap window
        segs_moss_diar.append(seg)

    if segs_moss_diar:
        all_results['MOSS-Audio'] = [
            {'speaker': s['speaker'], 'start': s['start'], 'end': s['end']}
            for s in segs_moss_diar
        ]
        # Store emotion labels separately for use in notebook 05
        moss_emotions = {i: s['emotion'] for i, s in enumerate(segs_moss_diar)}

        spk_ids = sorted(set(s['speaker'] for s in segs_moss_diar))
        print(f'\nMOSS-Audio diarization: {len(segs_moss_diar)} segs, {len(spk_ids)} speakers in {elapsed:.1f}s')
        print(f'Speakers: {spk_ids}')
        emotion_dist = pd.Series([s['emotion'] for s in segs_moss_diar]).value_counts().to_dict()
        print(f'Emotion distribution: {emotion_dist}')
    else:
        print('MOSS-Audio: no segments extracted. The model may not have produced parseable JSON.')
        print('Tip: try a simpler prompt or the 8B variant for better instruction following.')

except Exception as e:
    _is_oom = 'memory' in str(e).lower() or 'out of mem' in str(e).lower()
    if _is_oom:
        print('MOSS-Audio SKIPPED: not enough VRAM on this machine (needs ~10 GB).')
        print('Run this cell on GCP with a CUDA GPU — everything else continues fine.')
    else:
        print(f'MOSS-Audio diarization FAILED: {e}')
    try:
        import torch; torch.mps.empty_cache() if hasattr(torch.mps, 'empty_cache') else None
    except Exception: pass
    all_results['MOSS-Audio'] = []  # mark as skipped so downstream comparison still runs
    moss_emotions = {}

## Commercial — AssemblyAI, Deepgram, Speechmatics

Commercial APIs are worth testing as benchmarks against OSS — they typically train on much more diverse data and perform better on Indic content. All offer free tiers with no credit card required. Skip any you don't have a key for; the pipeline continues with whatever is available.

| API | Free tier | Indic strength |
|---|---|---|
| **AssemblyAI Universal-3** | Free tier (no CC) | Strong on Hindi; decent Tamil |
| **Deepgram Nova-3** | $200 credit (no CC) | Best word timestamps; good multilingual |
| **Speechmatics** | 480 min/month free | Good Indic; returns confidence per segment |

In [ ]:
# ── AssemblyAI Universal-3 ────────────────────────────────────────────────────
# pip install assemblyai
ASSEMBLYAI_API_KEY = os.getenv('ASSEMBLYAI_API_KEY') or getpass('AssemblyAI API key (Enter to skip): ')

if not ASSEMBLYAI_API_KEY.strip():
    print('[AssemblyAI] Skipped.')
else:
    try:
        import assemblyai as aai
        aai.settings.api_key = ASSEMBLYAI_API_KEY

        lang_code = SOURCE_LANGUAGE if SOURCE_LANGUAGE == 'hi' else None  # Tamil needs auto-detect
        config    = aai.TranscriptionConfig(speaker_labels=True, language_code=lang_code)

        t0 = time.time()
        transcript = aai.Transcriber().transcribe(VOCALS_WAV, config=config)
        elapsed    = time.time() - t0

        if transcript.error:
            raise RuntimeError(transcript.error)

        segs = []
        for utt in (transcript.utterances or []):
            segs.append({
                'speaker': f'SPEAKER_{utt.speaker}',
                'start':   utt.start / 1000.0,
                'end':     utt.end   / 1000.0,
            })
        all_results['AssemblyAI'] = sorted(segs, key=lambda x: x['start'])
        spk_count = len(set(s['speaker'] for s in segs))
        print(f'AssemblyAI: {len(segs)} segs, {spk_count} speakers, {elapsed:.1f}s  RTF={elapsed/DURATION:.2f}')
    except ImportError:
        print('assemblyai not installed. Run: pip install assemblyai')
    except Exception as e:
        print(f'AssemblyAI FAILED: {e}')

In [ ]:
# ── Deepgram Nova-3 ───────────────────────────────────────────────────────────
# pip install deepgram-sdk
DEEPGRAM_API_KEY = os.getenv('DEEPGRAM_API_KEY') or getpass('Deepgram API key (Enter to skip): ')

if not DEEPGRAM_API_KEY.strip():
    print('[Deepgram] Skipped.')
else:
    try:
        from deepgram import DeepgramClient, PrerecordedOptions

        dg_client = DeepgramClient(DEEPGRAM_API_KEY)
        with open(VOCALS_WAV, 'rb') as f:
            audio_data = {'buffer': f.read()}

        options = PrerecordedOptions(model='nova-3', diarize=True, language=SOURCE_LANGUAGE)
        t0      = time.time()
        resp    = dg_client.listen.rest.v('1').transcribe_file(audio_data, options)
        elapsed = time.time() - t0

        # Group consecutive words by speaker into turn segments
        words = resp.results.channels[0].alternatives[0].words or []
        segs  = []
        if words:
            cur_spk, seg_start, seg_end = words[0].speaker, words[0].start, words[0].end
            for w in words[1:]:
                if w.speaker == cur_spk:
                    seg_end = w.end
                else:
                    segs.append({'speaker': f'SPEAKER_{cur_spk:02d}', 'start': float(seg_start), 'end': float(seg_end)})
                    cur_spk, seg_start, seg_end = w.speaker, w.start, w.end
            segs.append({'speaker': f'SPEAKER_{cur_spk:02d}', 'start': float(seg_start), 'end': float(seg_end)})

        all_results['Deepgram-Nova3'] = sorted(segs, key=lambda x: x['start'])
        spk_count = len(set(s['speaker'] for s in segs))
        print(f'Deepgram Nova-3: {len(segs)} segs, {spk_count} speakers, {elapsed:.1f}s  RTF={elapsed/DURATION:.2f}')
    except ImportError:
        print('deepgram-sdk not installed. Run: pip install deepgram-sdk')
    except Exception as e:
        print(f'Deepgram FAILED: {e}')

In [ ]:
# ── Speechmatics ──────────────────────────────────────────────────────────────
# pip install speechmatics-python
# 480 min/month free tier. Strong on Indic content; returns per-word confidence.
SPEECHMATICS_API_KEY = os.getenv('SPEECHMATICS_API_KEY') or getpass('Speechmatics API key (Enter to skip): ')

if not SPEECHMATICS_API_KEY.strip():
    print('[Speechmatics] Skipped.')
else:
    try:
        import speechmatics
        from speechmatics.models import BatchTranscriptionConfig, ConnectionSettings

        settings = ConnectionSettings(url='https://asr.api.speechmatics.com/v2', auth_token=SPEECHMATICS_API_KEY)
        sm_client = speechmatics.client.BatchSpeechmaticsClient(settings)

        lang = SOURCE_LANGUAGE if SOURCE_LANGUAGE in ('hi', 'ta') else 'en'
        config = BatchTranscriptionConfig(
            language=lang,
            diarization='speaker',
            enable_entities=False,
        )

        t0 = time.time()
        with open(VOCALS_WAV, 'rb') as f:
            job_id = sm_client.submit_job(audio=(os.path.basename(VOCALS_WAV), f), transcription_config=config)
        print(f'  Speechmatics job: {job_id}. Waiting...')
        transcript_json = sm_client.wait_for_completion(job_id, transcription_format='json-v2')
        elapsed = time.time() - t0

        # Parse speaker-change events into turn segments
        results_json = transcript_json.get('results', [])
        segs, cur_spk, seg_start, seg_end = [], None, None, None
        for item in results_json:
            if item.get('type') != 'word': continue
            spk = item.get('alternatives', [{}])[0].get('speaker', 'UU')
            t_s = item['start_time']
            t_e = item['end_time']
            if cur_spk is None:
                cur_spk, seg_start, seg_end = spk, t_s, t_e
            elif spk == cur_spk:
                seg_end = t_e
            else:
                segs.append({'speaker': f'SPEAKER_{cur_spk}', 'start': float(seg_start), 'end': float(seg_end)})
                cur_spk, seg_start, seg_end = spk, t_s, t_e
        if cur_spk:
            segs.append({'speaker': f'SPEAKER_{cur_spk}', 'start': float(seg_start), 'end': float(seg_end)})

        all_results['Speechmatics'] = sorted(segs, key=lambda x: x['start'])
        spk_count = len(set(s['speaker'] for s in segs))
        print(f'Speechmatics: {len(segs)} segs, {spk_count} speakers, {elapsed:.1f}s  RTF={elapsed/DURATION:.2f}')
    except ImportError:
        print('speechmatics-python not installed. Run: pip install speechmatics-python')
    except Exception as e:
        print(f'Speechmatics FAILED: {e}')

## Metrics & Comparison


In [ ]:
rows = []
for name, segs in all_results.items():
    spk   = sorted(set(s['speaker'] for s in segs))
    total = sum(s['end'] - s['start'] for s in segs)
    rows.append({
        'Model':        name,
        'Segments':     len(segs),
        'Speakers':     len(spk),
        'Speaker IDs':  str(spk[:6]) + ('...' if len(spk) > 6 else ''),
        'Coverage %':   round(total / DURATION * 100, 1),
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))


In [ ]:
# ── Speaker timeline visualisation (first 60s) ────────────────────────────────
for name, segs in all_results.items():
    speakers  = sorted(set(s['speaker'] for s in segs))
    color_map = {sp: f'C{i}' for i, sp in enumerate(speakers)}
    fig, ax   = plt.subplots(figsize=(14, 1.5))
    for seg in segs:
        if seg['start'] > 60: break
        ax.barh(0, seg['end']-seg['start'], left=seg['start'],
                color=color_map[seg['speaker']], height=0.6, alpha=0.85)
    patches = [mpatches.Patch(color=color_map[sp], label=sp) for sp in speakers]
    ax.legend(handles=patches, loc='upper right', fontsize=7)
    ax.set_xlabel('Time (s)')
    ax.set_title(f'{name} — first 60s  ({len(speakers)} speakers, {len(segs)} segs)')
    ax.set_xlim(0, min(60, DURATION))
    ax.set_yticks([])
    plt.tight_layout()
    plt.show()


## Ensemble: DOVER-Lap voting

When >= 2 systems are available, DOVER-Lap can vote on a consensus diarization that typically improves DER by 5-15% over the best single system.


In [ ]:
# pip install dover-lap  (lightweight)
# dover-lap expects real RTTM files on disk, not in-memory objects.
import tempfile, shutil

try:
    try:
        from dover_lap import dover_lap as _dover_fn
        _dover_api = 'function'
    except ImportError:
        try:
            from dover_lap import DoverLap as _DoverLap
            _dover_api = 'class'
        except ImportError:
            raise ImportError('dover-lap not installed. Run: pip install dover-lap')

    if len(all_results) < 2:
        print('Need >= 2 diarization results for DOVER-Lap.')
    else:
        tmpdir = tempfile.mkdtemp()
        try:
            rttm_paths = []
            for name, segs in all_results.items():
                rp = os.path.join(tmpdir, f'{name}.rttm')
                with open(rp, 'w') as f:
                    for s in segs:
                        d = s['end'] - s['start']
                        f.write(f'SPEAKER file 1 {s["start"]:.3f} {d:.3f} <NA> <NA> {s["speaker"]} <NA> <NA>\n')
                rttm_paths.append(rp)

            out_rttm_tmp = os.path.join(tmpdir, 'dover_out.rttm')

            if _dover_api == 'function':
                _dover_fn(rttm_paths, out_rttm=out_rttm_tmp)
            else:
                dl = _DoverLap()
                dl.combine(rttm_paths, out_rttm=out_rttm_tmp)

            segs_dover = []
            with open(out_rttm_tmp) as f:
                for line in f:
                    p = line.strip().split()
                    if len(p) < 8 or p[0] != 'SPEAKER': continue
                    st = float(p[3]); d = float(p[4])
                    segs_dover.append({'speaker': p[7], 'start': st, 'end': st + d})

            all_results['DOVER-Lap'] = sorted(segs_dover, key=lambda x: x['start'])
            print(f'DOVER-Lap consensus: {len(segs_dover)} segments')
        finally:
            shutil.rmtree(tmpdir, ignore_errors=True)

except ImportError as e:
    print(e)
except Exception as e:
    print(f'DOVER-Lap FAILED: {e}')

In [ ]:
# ── Select winner & save ──────────────────────────────────────────────────────
# Priority: External (platform data) > DOVER-Lap > Pyannote-N > Pyannote
_n_key = f'Pyannote-3.1-N{NUM_SPEAKERS}' if NUM_SPEAKERS else None
_priority = ['External', 'DOVER-Lap'] + ([_n_key] if _n_key else []) + ['Pyannote-3.1', 'WhisperX']
for preferred in _priority:
    if preferred in all_results and all_results[preferred]:
        WINNER = preferred
        break
else:
    WINNER = next((k for k, v in all_results.items() if v), None)
    if WINNER is None:
        raise RuntimeError('No diarization results available — run at least one model cell first.')

# Manual override:
# WINNER = 'Pyannote-3.1'

final_segs = all_results[WINNER]
with open(OUT_JSON, 'w') as f:
    json.dump(final_segs, f, indent=2)

# RTTM
with open(OUT_RTTM, 'w') as f:
    for s in final_segs:
        d = s['end'] - s['start']
        f.write(f'SPEAKER file 1 {s["start"]:.3f} {d:.3f} <NA> <NA> {s["speaker"]} <NA> <NA>\n')

print(f'Winner: {WINNER}')
print(f'Saved {len(final_segs)} segments')
print(f'  JSON: {OUT_JSON}')
print(f'  RTTM: {OUT_RTTM}')